# 第 8 章：特徵值與特徵向量

本 Notebook 是 [`ch08_eigenvalues.md`](ch08_eigenvalues.md) 的精簡對照版本，程式碼邏輯與 [`ch08_eigenvalues.py`](ch08_eigenvalues.py) 一致，可互動執行。

## 學習目標

- 特徵值/特徵向量的定義 $Av = \lambda v$
- 特徵方程式 $\det(A-\lambda I)=0$ 的手算範例
- 特徵值性質：和 = trace(A)、積 = det(A)
- 對角化 $A = PDP^{-1}$
- 幾何意義：特徵向量方向不變，只被縮放

In [1]:
import os

import matplotlib
matplotlib.use("Agg")  # 無顯示器環境，僅存檔不顯示

import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams["font.sans-serif"] = [
    "PingFang TC", "PingFang SC", "Heiti TC", "Arial Unicode MS", "DejaVu Sans",
]
matplotlib.rcParams["axes.unicode_minus"] = False

OUT_DIR = os.getcwd()

## 1. 特徵值與特徵向量的定義：$Av = \lambda v$

用 `numpy.linalg.eig` 求矩陣 $A = \begin{bmatrix} 4 & 1 \\ 2 & 3 \end{bmatrix}$ 的特徵值與特徵向量。

In [2]:
A = np.array([[4.0, 1.0],
              [2.0, 3.0]])
print("矩陣 A =")
print(A)

eigenvalues, eigenvectors = np.linalg.eig(A)
print("\n特徵值 (eigenvalues) =", eigenvalues)
print("特徵向量矩陣 (每一行 column 對應一個特徵向量) =")
print(eigenvectors)

矩陣 A =
[[4. 1.]
 [2. 3.]]

特徵值 (eigenvalues) = [5.+0.j 2.+0.j]
特徵向量矩陣 (每一行 column 對應一個特徵向量) =
[[ 0.70710678+0.j -0.4472136 +0.j]
 [ 0.70710678+0.j  0.89442719+0.j]]


## 2. 驗證 $Av \approx \lambda v$

用 `np.allclose` 逐一驗證每個特徵值/特徵向量配對。

In [3]:
for i in range(len(eigenvalues)):
    lam = eigenvalues[i]
    v = eigenvectors[:, i]
    lhs = A @ v
    rhs = lam * v
    ok = np.allclose(lhs, rhs)
    print(f"lambda_{i+1} = {lam:.4f} | A@v = {lhs} | lambda*v = {rhs} | 驗證: {ok}")

lambda_1 = 5.0000+0.0000j | A@v = [3.53553391+0.j 3.53553391+0.j] | lambda*v = [3.53553391+0.j 3.53553391+0.j] | 驗證: True
lambda_2 = 2.0000+0.0000j | A@v = [-0.89442719+0.j  1.78885438+0.j] | lambda*v = [-0.89442719+0.j  1.78885438+0.j] | 驗證: True


## 3. 手算範例對照

手算過程（見 `.md` 文件）：特徵方程式 $\lambda^2 - 7\lambda + 10 = 0$，解得 $\lambda_1=5$（特徵向量 $(1,1)$）、$\lambda_2=2$（特徵向量 $(1,-2)$）。

In [4]:
hand_eigenvalues = np.array([5.0, 2.0])
hand_v1 = np.array([1.0, 1.0])
hand_v2 = np.array([1.0, -2.0])

print("手算特徵值：", sorted(hand_eigenvalues, reverse=True))
print("numpy 求得特徵值（排序後，取實部）：", sorted(eigenvalues.real, reverse=True))
print("兩者是否一致：", np.allclose(sorted(eigenvalues.real, reverse=True),
                                 sorted(hand_eigenvalues, reverse=True)))

print("\n驗證 A@v1 = 5*v1:", np.allclose(A @ hand_v1, 5 * hand_v1))
print("驗證 A@v2 = 2*v2:", np.allclose(A @ hand_v2, 2 * hand_v2))

手算特徵值： [np.float64(5.0), np.float64(2.0)]
numpy 求得特徵值（排序後，取實部）： [np.float64(5.0), np.float64(2.0)]
兩者是否一致： True

驗證 A@v1 = 5*v1: True
驗證 A@v2 = 2*v2: True


## 4. 特徵值的性質：和 = trace(A)，積 = det(A)

In [5]:
trace_A = np.trace(A)
det_A = np.linalg.det(A)
sum_eig = np.sum(eigenvalues)
prod_eig = np.prod(eigenvalues)

print("trace(A) =", trace_A, " | sum(特徵值) =", sum_eig,
      " | 相等:", np.allclose(trace_A, sum_eig))
print("det(A) =", det_A, " | prod(特徵值) =", prod_eig,
      " | 相等:", np.allclose(det_A, prod_eig))

trace(A) = 7.0  | sum(特徵值) = (7+0j)  | 相等: True
det(A) = 10.000000000000002  | prod(特徵值) = (10+0j)  | 相等: True


## 5. 對角化 $A = PDP^{-1}$

$P$ 由特徵向量組成、$D$ 為特徵值對角矩陣。驗證 $PDP^{-1} \approx A$。

In [6]:
P = eigenvectors
D = np.diag(eigenvalues)
P_inv = np.linalg.inv(P)

A_reconstructed = P @ D @ P_inv

print("P =\n", P)
print("D =\n", D)
print("\n重建 P@D@P^-1 =\n", A_reconstructed)
print("\n原矩陣 A =\n", A)

print("\nnp.allclose(P@D@P^-1, A) =", np.allclose(A_reconstructed, A))

rank_P = np.linalg.matrix_rank(P)
print(f"P 的秩 = {rank_P}，n = {A.shape[0]} -> 秩等於 n，A 可對角化。")

P =
 [[ 0.70710678+0.j -0.4472136 +0.j]
 [ 0.70710678+0.j  0.89442719+0.j]]
D =
 [[5.+0.j 0.+0.j]
 [0.+0.j 2.+0.j]]

重建 P@D@P^-1 =
 [[4.+0.j 1.+0.j]
 [2.+0.j 3.+0.j]]

原矩陣 A =
 [[4. 1.]
 [2. 3.]]

np.allclose(P@D@P^-1, A) = True
P 的秩 = 2，n = 2 -> 秩等於 n，A 可對角化。


### 反例：不可對角化的矩陣

$B = \begin{bmatrix} 2 & 1 \\ 0 & 2 \end{bmatrix}$ 的特徵值 $\lambda=2$ 為重根，但只有 1 個線性獨立的特徵向量，因此不可對角化。

In [7]:
B = np.array([[2.0, 1.0],
              [0.0, 2.0]])
eig_B, vec_B = np.linalg.eig(B)
rank_vec_B = np.linalg.matrix_rank(vec_B)
print("特徵值 =", eig_B)
print("特徵向量矩陣秩 =", rank_vec_B, "(< 2，不可對角化)")

特徵值 = [2.+0.j 2.+0.j]
特徵向量矩陣秩 = 1 (< 2，不可對角化)


## 6. 幾何意義

左圖：一般向量經 $A$ 變換後方向改變。右圖：特徵向量方向不變，只被縮放 $\lambda$ 倍。

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

ax = axes[0]
general_vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0])]
colors_gen = ["tab:blue", "tab:orange", "tab:purple"]
origin = np.array([0.0, 0.0])
for v, c in zip(general_vectors, colors_gen):
    Av = A @ v
    ax.quiver(*origin, *v, angles="xy", scale_units="xy", scale=1,
              color=c, alpha=0.4, width=0.01)
    ax.quiver(*origin, *Av, angles="xy", scale_units="xy", scale=1,
              color=c, width=0.01)
ax.set_xlim(-2, 6)
ax.set_ylim(-2, 6)
ax.set_aspect("equal")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.grid(True, linestyle="--", alpha=0.5)
ax.set_title("一般向量：方向改變")
ax.set_xlabel("x")
ax.set_ylabel("y")

ax2 = axes[1]
eig_dirs = [hand_v1 / np.linalg.norm(hand_v1), hand_v2 / np.linalg.norm(hand_v2)]
eig_lams = [5.0, 2.0]
colors_eig = ["tab:green", "tab:red"]
for v, lam, c in zip(eig_dirs, eig_lams, colors_eig):
    Av = A @ v
    ax2.quiver(*origin, *v, angles="xy", scale_units="xy", scale=1,
               color=c, alpha=0.4, width=0.012, label=f"v (lambda={lam:.0f})")
    ax2.quiver(*origin, *Av, angles="xy", scale_units="xy", scale=1,
               color=c, width=0.012, label=f"A v = {lam:.0f} v")
ax2.set_xlim(-3, 6)
ax2.set_ylim(-3, 6)
ax2.set_aspect("equal")
ax2.axhline(0, color="gray", linewidth=0.5)
ax2.axvline(0, color="gray", linewidth=0.5)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.set_title("特徵向量：方向不變，只被縮放")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.legend(loc="upper left", fontsize=8)

fig_path = os.path.join(OUT_DIR, "eigenvectors_geometry.png")
plt.savefig(fig_path, dpi=120, bbox_inches="tight")
print("已儲存圖片至:", fig_path)
fig

已儲存圖片至: /Users/rexwang/workspace/linear-algebra-with-matlab-python-tutorial/ch08_eigenvalues/eigenvectors_geometry.png


<Figure size 1200x600 with 2 Axes>

## 重點整理

- $Av = \lambda v$（$v \neq 0$）定義特徵值與特徵向量。
- 特徵方程式 $\det(A-\lambda I)=0$ 求特徵值，代回求特徵向量。
- $\sum \lambda_i = \mathrm{trace}(A)$，$\prod \lambda_i = \det(A)$。
- $A = PDP^{-1}$：需要 $n$ 個線性獨立的特徵向量才能對角化。
- 特徵向量是矩陣變換下方向不變、只被縮放的向量。

詳細教學說明請見 [`ch08_eigenvalues.md`](ch08_eigenvalues.md)。